# 07 · Retrieval evaluation (step 22a: BM25 baseline)

**Question:** when someone asks something, does the right section reach the top 5?

* Answer key is at **section level** (`5.2`, `Annexure I`, `Letter 5`), not chunk level, so the same questions can score any chunking strategy later.
* A hit inside a **sub-section** counts for its parent (expected `5.3` accepts a chunk from `5.3.7`).
* Metrics: **hit@1**, **hit@5** (main number), **MRR@10** (1 / rank of the first correct chunk).

One-time install (both pure Python, no DLLs): `pip install rank_bm25 snowballstemmer`

Put `questions.csv` in `data/eval/` before running.

##  07.0 Setup + load 

In [ ]:

import sys, json, re
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.paths import SECTIONS, CHUNKS


from rank_bm25 import BM25Okapi
import snowballstemmer

EVAL_DIR  = CHUNKS.parents[1] / "eval"            # data/interim/chunks.jsonl -> data/eval
QUESTIONS = EVAL_DIR / "questions.csv"
RESULTS   = EVAL_DIR / "bm25_results.csv"         # one row per (variant, question)
RUNS_LOG  = EVAL_DIR / "runs.csv"                 # one row per variant per run: the scoreboard over time
K, DEPTH  = 5, 10                                  # hit@5 is the main number; look 10 deep for MRR
 
read_jsonl = lambda p: [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
SECS, CH = read_jsonl(SECTIONS), read_jsonl(CHUNKS)
SEC = {s["section_id"]: s for s in SECS}

CHILDREN = defaultdict(list)
for s in SECS:
    if s["parent_id"]:
        CHILDREN[s["parent_id"]].append(s["section_id"])

def with_descendants(ids):
    out, todo = set(), list(ids)
    while todo:
        sid = todo.pop()
        if sid not in out:
            out.add(sid)
            todo += CHILDREN.get(sid, [])
    return out

def key_of(c):                                     # the unit we score: a section id, or "L05" for letter 5
    return f"L{int(c['letter_no']):02d}" if c.get("letter_no") is not None else c["section_id"]

LABEL = {s["section_id"]: s["path"][-1] for s in SECS}
LABEL.update({key_of(c): c["path"][-1] for c in CH if c.get("letter_no") is not None})
LETTER_KEYS = {key_of(c) for c in CH if c.get("letter_no") is not None}

print(f"{len(SECS)} sections, {len(CH)} chunks, {len(LETTER_KEYS)} letters")

106 sections, 466 chunks, 21 letters


## 07.1 Questions → answer key

Each row needs **`expected`** or **`answer_phrase`** (or both):

| `expected` written as | means |
|---|---|
| `5.2` | the section numbered 5.2 (and its sub-sections) |
| `Letter 5` | letter 5 |
| `Annexure I` | any section whose label/title contains these exact words (`Annexure I` does **not** match `Annexure II`) |
| `Annexure I > Written` | a section matching `Written` that sits under `Annexure I` |
| `5.3; 5.4` | either is correct |

`answer_phrase` = a short exact phrase copied from the PDF. Without `expected`, the key becomes whatever section contains that phrase. With both, the tests check the phrase really is inside the expected section.

All 30 rows were written from the PDF itself and every `answer_phrase` was checked against the PDF text. If one still fails, it is usually a small text difference in our pipeline (a hyphen, a quote mark); fix it with `toc()` / `find()` below.

## 07.1 Questions → answer key

In [41]:
# ===================== 07.1 Questions → answer key =====================
TYPES = {"keyword", "paraphrase", "acronym", "table", "letter", "unreadable"}
flat  = lambda s: re.sub(r"\s+", " ", str(s)).strip().lower()
word  = lambda p: re.compile(r"(?<!\w)" + re.escape(p.strip()) + r"(?!\w)", re.I)   # whole words only

def section_matches(s, pieces):
    own = f"{s.get('label') or ''} | {s.get('title') or ''}"
    if not word(pieces[-1]).search(own):
        return False
    above, pos = " > ".join(s["path"][:-1]), 0
    for p in pieces[:-1]:                          # 'Annexure I > Written': earlier pieces must be ancestors, in order
        m = word(p).search(above, pos)
        if not m:
            return False
        pos = m.end()
    return True

def by_expected(expected):
    keys, missing = set(), []
    for part in [p.strip() for p in expected.split(";") if p.strip()]:
        m = re.fullmatch(r"(?i)letter\s*(\d+)", part)
        if m:
            hit = {f"L{int(m[1]):02d}"} & LETTER_KEYS
        elif re.fullmatch(r"\d+(\.\d+)*", part):
            hit = {s["section_id"] for s in SECS if str(s.get("number") or "").rstrip(".") == part}
        else:
            hit = {s["section_id"] for s in SECS if section_matches(s, part.split(">"))}
        keys |= hit
        if not hit:
            missing.append(part)
    return keys, missing

def by_phrase(phrase):
    return {key_of(c) for c in CH if flat(phrase) in flat(c["text"])}

def resolve(row):
    problems, keys = [], set()
    if row.type not in TYPES:
        problems.append(f"unknown type '{row.type}'")
    if row.expected:
        keys, missing = by_expected(row.expected)
        problems += [f"expected '{m}' not found" for m in missing]
    if row.answer_phrase:
        found = by_phrase(row.answer_phrase)
        if not found:
            problems.append("answer_phrase not found in any chunk")
        elif row.expected and not found & with_descendants(keys):
            problems.append(f"phrase found in {sorted(found)}, not in expected {sorted(keys)}")
        if not row.expected:
            keys = found
    if not row.expected and not row.answer_phrase:
        problems.append("needs expected or answer_phrase")
    return sorted(keys), "; ".join(problems)

Q = pd.read_csv(QUESTIONS, dtype=str, keep_default_na=False)
resolved     = [resolve(r) for r in Q.itertuples()]
Q["keys"]    = [keys for keys, _ in resolved]
Q["problem"] = [problem for _, problem in resolved]
Q["targets"]   = [with_descendants(k) for k in Q["keys"]]          # what counts as a correct chunk
Q["key_label"] = [" | ".join(LABEL.get(k, k) for k in ks)[:90] for ks in Q["keys"]]

print(f"{len(Q)} questions:", dict(Counter(Q["type"])))
print(f"{(Q['problem'] != '').sum()} with a problem (fix these first)")
Q[["qid", "type", "expected", "keys", "key_label", "problem"]]

42 questions: {'keyword': 10, 'table': 2, 'paraphrase': 14, 'acronym': 12, 'letter': 3, 'unreadable': 1}
7 with a problem (fix these first)


,qid,type,expected,keys,key_label,problem
0,Q01,keyword,Chapter 1: Introduction,[S002],Chapter 1: Introduction,answer_phrase not found in any chunk
1,Q02,table,5.3,[S023],5.3 Process of Recruitment,
2,Q03,paraphrase,,[S020],Chapter 5: Recruitment,
3,Q04,keyword,5.3.3,[S026],5.3.3 Advertisement for Vacancy,answer_phrase not found in any chunk
4,Q05,paraphrase,5.3.3,[S026],5.3.3 Advertisement for Vacancy,
5,Q06,paraphrase,5.3.7,[S030],5.3.7 Reference Check,
6,Q07,keyword,8.10.1,[S065],8.10.1 Human Resource Information Management S...,answer_phrase not found in any chunk
7,Q08,paraphrase,4.3,[S015],4.3 Engagement through Deputation,
8,Q09,keyword,9.1,[S067],9.1 Contents of Code of Conduct Policy,answer_phrase not found in any chunk
9,Q10,paraphrase,6.1.2,[S035],6.1.2 Issuance of Identity Card,


## 07.2 Fixing the answer key
`toc("appraisal")` lists sections whose path contains the word; `find("phrase")` shows which chunks contain a phrase. Edit `questions.csv`, then re-run **07.1**.

In [42]:
# ===================== 07.2 Helpers for fixing questions.csv =====================
def toc(text=None):
    for s in SECS:
        line = " > ".join(s["path"])
        if text is None or text.lower() in line.lower():
            print(f"{s['section_id']}  {str(s.get('number') or ''):8} p.{str(s.get('printed_start') or ''):5} {'  ' * s['level']}{s['path'][-1]}")

def find(phrase, n=8):
    hits = [c for c in CH if flat(phrase) in flat(c["text"])]
    print(f"'{phrase}' in {len(hits)} chunks")
    for c in hits[:n]:
        i = flat(c["text"]).find(flat(phrase))
        print(f"  {c['chunk_id']:10} key={key_of(c):6} p.{c['printed_pages']}  {LABEL.get(key_of(c), '')[:50]}")
        print(f"      …{flat(c['text'])[max(0, i - 60): i + 90]}…")

toc("leave")
find("leave policy")

S052  8.4      p.23      8.4 Leave Policy
S053  8.4.1    p.23        8.4.1 Public Holidays
S085           p.78      Leave Policy
'leave policy' in 3 chunks
  S052-01    key=S052   p.['23']  8.4 Leave Policy
      …8.4. leave policy the state nhm leave policy should be applicable to all nhm staff. if the stat…
  S076-10    key=S076   p.['56']  Contents of the Contract
      … paternity leave as per the terms and conditions of shs/dhs leave policy. this is in addition to one day weekly off to be fixed by the shs/dhs. 14. yo…
  S085-01    key=S085   p.['78']  Leave Policy
      …leave policy process of applying leave through hrmis a proper leave management module shou…


## 07.3 BM25
Same chunks, four ways of turning text into words:
* **field**: `text` only, or `embed_text` (section path + sub-heading + text)
* **stem/stop**: `committees → committe`, `appraised/appraisal → apprais`; drop words like *the, of, what*

`k1=1.5, b=0.75` are the library defaults; we only tune them if the numbers ask for it.

In [43]:
# ===================== 07.3 Tokenizer + BM25 =====================
STOP = set("""a an the of to in on for and or is are was were be been by with as at from that this these those it its
what which who whom how why when where does do did should shall can could will would may might must about into than
there their they them his her he she we our you your i my me not no any all""".split())
_stem = snowballstemmer.stemmer("english").stemWord

def tokenize(text, stem=True, stop=True):
    words = re.findall(r"[a-z0-9]+", text.lower())
    if stop:
        words = [w for w in words if w not in STOP]
    return [_stem(w) for w in words] if stem else words

def build_bm25(field="embed_text", stem=True, stop=True, k1=1.5, b=0.75):
    bm = BM25Okapi([tokenize(c[field], stem, stop) or ["_empty_"] for c in CH], k1=k1, b=b)
    def search(question, k=DEPTH):
        scores = bm.get_scores(tokenize(question, stem, stop))
        top = sorted(range(len(CH)), key=lambda i: -scores[i])[:k]
        return [(CH[i], float(scores[i])) for i in top]
    return search

print(tokenize("What are the Recruitment Committees' duties?"))
print([(c["chunk_id"], round(s, 2)) for c, s in build_bm25()("recruitment committee members", 3)])

['recruit', 'committe', 'duti']
[('S022-01', 13.78), ('S069-01', 9.22), ('S106-05', 7.53)]


In [44]:
# ===================== 07.4 Metrics =====================
def first_hit_rank(results, targets):
    for rank, (c, _) in enumerate(results, 1):
        if key_of(c) in targets:
            return rank
    return None                                    # not in the top DEPTH

def evaluate(search, questions=None):
    rows = []
    for q in (Q if questions is None else questions).itertuples():
        if not q.keys:                             # unresolved question: no answer key, can't score
            continue
        res = search(q.question)
        r = first_hit_rank(res, q.targets)
        rows.append({"qid": q.qid, "type": q.type, "question": q.question, "rank": r,
                     "hit@1": r == 1, "hit@5": r is not None and r <= K, "rr": 1 / r if r else 0.0,
                     "top1": f"{res[0][0]['chunk_id']} {LABEL.get(key_of(res[0][0]), '')[:40]}"})
    return pd.DataFrame(rows)

def summary(df, name):
    return {"variant": name, "n": len(df), "hit@1": round(df["hit@1"].mean(), 3),
            "hit@5": round(df["hit@5"].mean(), 3), "mrr@10": round(df["rr"].mean(), 3)}

In [45]:
# ===================== 07.5 Run the variants =====================
VARIANTS = {
    "A text, plain words":      dict(field="text",       stem=False, stop=False),
    "B text, stem+stop":        dict(field="text",       stem=True,  stop=True),
    "C path+text, plain words": dict(field="embed_text", stem=False, stop=False),
    "D path+text, stem+stop":   dict(field="embed_text", stem=True,  stop=True),
}
SEARCH = {name: build_bm25(**cfg) for name, cfg in VARIANTS.items()}
RUNS   = {name: evaluate(s) for name, s in SEARCH.items()}
BOARD  = pd.DataFrame([summary(df, name) for name, df in RUNS.items()]).set_index("variant")
BEST   = BOARD.sort_values(["hit@5", "mrr@10"]).index[-1]
print("best:", BEST)
display(BOARD)

by_type = RUNS[BEST].groupby("type").agg(n=("qid", "size"), hit_at_5=("hit@5", "mean"), mrr=("rr", "mean")).round(2)
by_type.sort_values("hit_at_5")

best: D path+text, stem+stop


,n,hit@1,hit@5,mrr@10
variant,,,,
"A text, plain words",39,0.359,0.692,0.484
"B text, stem+stop",39,0.538,0.718,0.616
"C path+text, plain words",39,0.436,0.718,0.540
"D path+text, stem+stop",39,0.641,0.769,0.695


,n,hit_at_5,mrr
type,,,
acronym,12,0.58,0.57
paraphrase,11,0.64,0.52
keyword,10,1.00,0.85
letter,3,1.00,1.00
table,2,1.00,1.00
unreadable,1,1.00,1.00


## 07.6 Misses — read these, they tell us what to do next
* right section at rank 6–10 → close; hybrid/reranking usually fixes
* not found, **paraphrase** → no shared words; this is exactly what embeddings are for
* not found, **keyword** → look at the top-1: wrong answer key, or a chunking problem

In [46]:
# ===================== 07.6 Misses of the best variant =====================
def show_misses(name=BEST, n_top=3):
    df = RUNS[name]
    miss = df[~df["hit@5"]]
    print(f"{name}: {len(miss)} misses of {len(df)}\n")
    for m in miss.itertuples():
        q = Q[Q["qid"] == m.qid].iloc[0]
        print(f"{m.qid} [{m.type}] {m.question}")
        print(f"   expected: {q.key_label}   rank: {int(m.rank) if pd.notna(m.rank) else f'not in top {DEPTH}'}")
        for rank, (c, s) in enumerate(SEARCH[name](m.question, n_top), 1):
            print(f"   {rank}. {s:5.1f} {c['chunk_id']:10} {LABEL.get(key_of(c), '')[:45]:45} | {flat(c['text'])[:70]}")
        print()

show_misses()

D path+text, stem+stop: 9 misses of 39

Q08 [paraphrase] Can people already in government service work for NHM?
   expected: 4.3 Engagement through Deputation   rank: not in top 10
   1.   9.9 L04-03     Letter 4 (3rd February, 2016): Strengthening  | health centers. ! measures such as offering competitive and negotiable
   2.   8.7 S099-01    Person-Centred Care through Shared leadership | person-centred care through shared leadership- banka model of bihar in
   3.   7.4 S073-43    Sample ToRs/ Roles and Responsibilities       | - he/she will monitor the implementation of iec/bcc activities in the 

Q14 [paraphrase] Can new mothers working under NHM get time off?
   expected: 8.4 Leave Policy   rank: not in top 10
   1.  13.1 S037-01    6.2 Orientation                               | 6.2. orientation new recruits should get a brief orientation to their 
   2.   9.8 S080-04    Expenses Policy                               | - if travelling for more than 5 days at a stretch, an employee

In [ ]:
# # ===================== 07.T Tests =====================
# n_pass, report = -1, []                 # a crash below can never leave an old "all passed" behind
# TESTS = []
# def check(group, name, get_actual, expected, desc=None):
#     try:
#         actual = get_actual()
#         ok = expected(actual) if callable(expected) else actual == expected
#     except Exception as e:
#         actual, ok = f"ERROR {type(e).__name__}: {e}", False
#     TESTS.append({"group": group, "test": name, "result": "PASS" if ok else "FAIL",
#                   "expected": desc if callable(expected) else expected, "actual": actual})

# G = "07 questions"
# check(G, "25-30 questions", lambda: len(Q), lambda n: 25 <= n <= 30, "25..30")
# check(G, "qid unique", lambda: [k for k, v in Counter(Q["qid"]).items() if v > 1], [])
# check(G, "every question ends with '?'", lambda: list(Q.loc[~Q["question"].str.strip().str.endswith("?"), "qid"]), [])
# check(G, "types valid", lambda: sorted(set(Q["type"]) - TYPES), [])
# check(G, "at least 6 paraphrase questions (the hard case for BM25)", lambda: int((Q["type"] == "paraphrase").sum()),
#       lambda n: n >= 6, ">= 6")
# check(G, "every question resolves, no problems",
#       lambda: [(r.qid, r.problem) for r in Q.itertuples() if r.problem], [])
# check(G, "answer key not too loose (<= 3 sections per question)",
#       lambda: [(r.qid, len(r.keys)) for r in Q.itertuples() if len(r.keys) > 3], [])
# check(G, "all questions scored", lambda: {name: len(df) for name, df in RUNS.items()},
#       lambda d: set(d.values()) == {len(Q)}, f"{len(Q)} per variant")

# G = "07 resolver unit"
# check(G, "'Letter 99' does not resolve", lambda: by_expected("Letter 99"), (set(), ["Letter 99"]))
# check(G, "whole words: 'Annexure I' never matches 'Annexure II'",
#       lambda: [SEC[k]["path"][-1] for k in by_expected("Annexure I")[0] if re.search(r"Annexure I[IV]", SEC[k]["path"][-1])], [])
# check(G, "a sub-section counts for its parent",
#       lambda: all(c in with_descendants({p}) for p, cs in CHILDREN.items() for c in cs), True)

# G = "07 metric unit"
# _r = [({"letter_no": None, "section_id": x}, 0.0) for x in ["S1", "S2", "S3"]]
# check(G, "rank of first correct", lambda: (first_hit_rank(_r, {"S2"}), first_hit_rank(_r, {"S9"})), (2, None))
# check(G, "letter chunks scored by letter", lambda: key_of({"letter_no": 5, "section_id": None}), "L05")
# _df = pd.DataFrame({"hit@1": [True, False], "hit@5": [True, True], "rr": [1.0, 0.25]})
# check(G, "summary maths", lambda: summary(_df, "t"), {"variant": "t", "n": 2, "hit@1": 0.5, "hit@5": 1.0, "mrr@10": 0.625})
# check(G, "hit@5 >= hit@1 and 0 <= mrr <= 1 for every variant",
#       lambda: [v for v, r in BOARD.iterrows() if not (r["hit@5"] >= r["hit@1"] and 0 <= r["mrr@10"] <= 1)], [])

# G = "07 bm25 sanity"
# check(G, "stemming: 'Recruitment Committees' == 'recruitment committee'",
#       lambda: tokenize("Recruitment Committees") == tokenize("recruitment committee"), True)
# check(G, "stop words dropped", lambda: tokenize("What is the leave policy of the State?"), ["leav", "polici", "state"])
# check(G, "no chunk without words", lambda: [c["chunk_id"] for c in CH if not tokenize(c["text"])], [])
# def self_retrieval(name=BEST, every=7):
#     probe = [c for c in CH[::every] if c["n_chars"] >= 200]
#     if not probe:
#         return "no chunk long enough to probe"
#     ok = sum(key_of(SEARCH[name](c["text"][:200], 1)[0][0]) == key_of(c) for c in probe)
#     return round(ok / len(probe), 2)
# check(G, "a chunk's own first 200 chars finds its section at rank 1 (>= 90%)", self_retrieval, lambda x: isinstance(x, float) and x >= 0.9, ">= 0.9")

# report = pd.DataFrame(TESTS)
# n_pass = int((report["result"] == "PASS").sum())
# print(f"{'✅' if n_pass == len(report) else '❌'}  {n_pass}/{len(report)} tests passed")
# failed = report[report["result"] == "FAIL"]
# failed if len(failed) else "All tests passed — nothing to show." 

❌  15/19 tests passed


,group,test,result,expected,actual
0,07 questions,25-30 questions,FAIL,25..30,42
2,07 questions,every question ends with '?',FAIL,[],"[Q09, Q41, Q42]"
5,07 questions,"every question resolves, no problems",FAIL,[],"[(Q01, answer_phrase not found in any chunk), ..."
7,07 questions,all questions scored,FAIL,42 per variant,"{'A text, plain words': 39, 'B text, stem+stop..."


In [ ]:
# # ===================== 07.H Save (only if all tests passed) =====================
# assert n_pass == len(report), "Tests failed — fix them before saving."
# stamp = datetime.now().strftime("%Y-%m-%d %H:%M")

# long = pd.concat([df.assign(variant=name) for name, df in RUNS.items()], ignore_index=True)
# long.to_csv(RESULTS, index=False, encoding="utf-8")

# log = BOARD.reset_index().assign(run=stamp, retriever="bm25", chunks=CHUNKS.name, n_chunks=len(CH))
# log.to_csv(RUNS_LOG, mode="a", header=not RUNS_LOG.exists(), index=False, encoding="utf-8")

# back = pd.read_csv(RESULTS)
# assert len(back) == len(long) and set(back["variant"]) == set(VARIANTS)
# assert (pd.read_csv(RUNS_LOG)["run"] == stamp).sum() == len(VARIANTS)
# print(f"saved {RESULTS.name} ({len(back)} rows) and appended {len(VARIANTS)} rows to {RUNS_LOG.name}")

In [49]:
# ===================== 07.8 Embeddings: chunks → vectors =====================
import numpy as np
from fastembed import TextEmbedding

model = TextEmbedding("BAAI/bge-small-en-v1.5")      # first run downloads ~67 MB
VEC = np.array(list(model.passage_embed([c["embed_text"] for c in CH])))
print(VEC.shape)                                     # (439, 384): 384 numbers per chunk

(466, 384)


In [50]:
model = TextEmbedding("BAAI/bge-small-en-v1.5", cache_dir=str(ROOT / "models"))

In [ ]:
# ===================== 07.9 Vector search =====================
def search_vec(question, k=DEPTH):
    q = np.array(list(model.query_embed(question)))[0]   # question → 384 numbers
    scores = VEC @ q                                     # closeness to every chunk (1 = same meaning)
    top = np.argsort(-scores)[:k]                        # rank: highest first
    return [(CH[i], float(scores[i])) for i in top]

for c, s in search_vec("Can new mothers working under NHM get time off?", 3):
    print(round(s, 3), c["chunk_id"], LABEL.get(key_of(c), "")[:50])

0.76 S052-01 8.4 Leave Policy
0.703 S085-03 Leave Policy
0.684 S076-10 Contents of the Contract


In [51]:
# ===================== 07.10 Compare =====================
SEARCH["E embeddings"] = search_vec
RUNS["E embeddings"]   = evaluate(search_vec)
display(pd.DataFrame([summary(df, n) for n, df in RUNS.items()]).set_index("variant"))
show_misses("E embeddings")

,n,hit@1,hit@5,mrr@10
variant,,,,
"A text, plain words",39,0.359,0.692,0.484
"B text, stem+stop",39,0.538,0.718,0.616
"C path+text, plain words",39,0.436,0.718,0.540
"D path+text, stem+stop",39,0.641,0.769,0.695
E embeddings,39,0.564,0.667,0.639


E embeddings: 13 misses of 39

Q05 [paraphrase] Where should a short job notice be printed?
   expected: 5.3.3 Advertisement for Vacancy   rank: 9
   1.   0.7 S034-01    6.1.1 Appointment / Offer Letter              | 6.1.1. appointment / offer letter on the day of joining, the new emplo
   2.   0.7 S089-02    Exit Policy                                   | letter of acceptance on resignation/ letter of end of contract - accep
   3.   0.7 S076-02    Contents of the Contract                      | - agreement on the probation period - specific deliverables during the

Q08 [paraphrase] Can people already in government service work for NHM?
   expected: 4.3 Engagement through Deputation   rank: 10
   1.   0.7 S017-01    4.5 Engagement of Retired Government Official | 4.5. engagement of retired government officials there is also the prov
   2.   0.7 S007-01    2.1.3 Support Service                         | 2.1.3. support service the hr under support services include data entr
   3.   0.7 

In [52]:
# ===================== 07.11 Hybrid (BM25 + embeddings, RRF) =====================
def search_hybrid(question, k=DEPTH, n=50, C=60):
    score = {}
    for search in (SEARCH["D path+text, stem+stop"], search_vec):
        for rank, (c, _) in enumerate(search(question, n), 1):       # top 50 from each
            score[c["chunk_id"]] = score.get(c["chunk_id"], 0) + 1 / (C + rank)
    by_id = {c["chunk_id"]: c for c in CH}
    top = sorted(score, key=score.get, reverse=True)[:k]
    return [(by_id[i], score[i]) for i in top]

SEARCH["F hybrid"] = search_hybrid
RUNS["F hybrid"]   = evaluate(search_hybrid)
display(pd.DataFrame([summary(df, n) for n, df in RUNS.items()]).set_index("variant"))
show_misses("F hybrid")

,n,hit@1,hit@5,mrr@10
variant,,,,
"A text, plain words",39,0.359,0.692,0.484
"B text, stem+stop",39,0.538,0.718,0.616
"C path+text, plain words",39,0.436,0.718,0.540
"D path+text, stem+stop",39,0.641,0.769,0.695
E embeddings,39,0.564,0.667,0.639
F hybrid,39,0.667,0.846,0.730


F hybrid: 6 misses of 39

Q08 [paraphrase] Can people already in government service work for NHM?
   expected: 4.3 Engagement through Deputation   rank: 6
   1.   0.0 S017-01    4.5 Engagement of Retired Government Official | 4.5. engagement of retired government officials there is also the prov
   2.   0.0 S073-44    Sample ToRs/ Roles and Responsibilities       | - he/she will develop strategies or plans to improve the quality of se
   3.   0.0 S073-43    Sample ToRs/ Roles and Responsibilities       | - he/she will monitor the implementation of iec/bcc activities in the 

Q12 [keyword] What happens to staff who do not meet the minimum performance benchmarks?
   expected: 8.2 Performance Appraisal   rank: 6
   1.   0.0 L20-03     Letter 20 (15th April, 2019): Minimum Perform | every nodal under nhm would have to achieve the minimum performance be
   2.   0.0 L21-02     Letter 21 (13th July, 2019): Reviewing the Pe | i. review the performance indicators in consultation with the progra

In [53]:
from fastembed import TextEmbedding


documents = [
    "FastEmbed is lighter than Transformers & Sentence-Transformers.",
    "FastEmbed is supported by and maintained by Qdrant.",
]

embedding_model = TextEmbedding()
print("The model BAAI/bge-small-en-v1.5 is ready to use.")

The model BAAI/bge-small-en-v1.5 is ready to use.


In [54]:
embeddings_generator = embedding_model.embed(documents)
embeddings_list = list(embeddings_generator)
len(embeddings_list[0])  

384

In [55]:
print("Embeddings:\n", embeddings_list)

Embeddings:
 [array([-9.47912633e-02,  1.00905541e-02, -3.08777597e-02,  2.37377789e-02,
        2.35279743e-03,  6.29410788e-04, -8.24827850e-02,  8.80842097e-04,
        3.72184925e-02,  1.43865319e-02, -2.38843076e-02,  8.91665288e-04,
        3.01584341e-02,  4.09216918e-02,  4.37190719e-02,  8.65856092e-03,
        5.05659953e-02, -1.26748011e-02, -3.14105935e-02,  2.05807332e-02,
        8.13105479e-02, -5.42159118e-02,  1.01771392e-02, -4.59036939e-02,
       -7.67947733e-03, -3.52470055e-02, -2.36045700e-02, -2.69214641e-02,
       -2.74143368e-02, -1.57292753e-01,  1.25282714e-02, -3.55667062e-02,
        6.45262599e-02, -1.34957000e-03, -3.40481251e-02,  9.80415568e-03,
       -3.24762650e-02,  1.50259333e-02, -6.96414709e-02, -1.67709664e-02,
       -2.34447196e-02,  2.00079363e-02, -1.56253725e-02,  1.40285341e-03,
       -1.32675795e-02, -2.17130072e-02,  6.85757995e-02, -2.02343911e-02,
       -9.09148902e-03, -1.49060460e-02,  6.37270063e-02,  1.35739595e-02,
        7.0